# Setup

In [ ]:
import datasets
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download
from pprint import pprint

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

utils.py: 0.00B [00:00, ?B/s]

## Preprocessing



In [ ]:
raw_dataset = datasets.load_dataset("sookiemonster/asrs-narratives")

train_ds = datasets.load_dataset("sookiemonster/asrs-narratives-rebalance", split='train')
valid_ds = raw_dataset['validation']
test_ds = raw_dataset['test']

labels = train_ds.features['label'].names

id_to_label = { idx : label for idx, label in enumerate(labels) }
label_to_id = { label : idx for idx, label in id_to_label.items() }

data/train-00000-of-00001.parquet:   0%|          | 0.00/22.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22992 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [ ]:
from functools import partial

def filter_labels(ds, to_remove:list):
  to_remove_set = set(to_remove)
  return ds.filter(lambda example : id_to_label[example['label']] not in to_remove_set)

filter_ambiguous = partial(filter_labels, to_remove=['ambiguous'])

filtered_train_ds = filter_ambiguous(train_ds)
filtered_valid_ds = filter_ambiguous(valid_ds)
filtered_test_ds = filter_ambiguous(test_ds)

Filter:   0%|          | 0/22992 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4441 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [ ]:
def _validate_groupings(groupings:dict[str, set]):
  mut_excl = [va.isdisjoint(vb) for ka, va in groupings.items() for kb, vb in groupings.items() if ka != kb]
  assert all(mut_excl), f"{mut_excl}"

  all_labels = set([label for val_set in groupings.values() for label in val_set])
  assert all_labels == set(id_to_label.values()), f"Missing: {set(id_to_label.values()) - all_labels}"


def group_labels(ds, groupings:dict[str, set]):
  _validate_groupings(groupings)
  group_names = list(groupings.keys())
  group_names.sort()

  fine_grained_label_to_group = {
      label : group_name for group_name, val_set in groupings.items() for label in val_set
  }

  res = ds.map(lambda ex: {"group" : fine_grained_label_to_group[ id_to_label[ex['label']] ]})
  res = res.filter(lambda ex: ex["group"] != 'DELETE')

  new_features = res.features.copy()
  group_names.remove("DELETE")
  new_features["group"] = ClassLabel(names=group_names)

  res = res.cast(new_features)
  return res

In [ ]:
def get_inv_class_weights(ds, labels, filter_by='group'):
  counts = ds.to_pandas()[filter_by].value_counts().to_dict()

  weights = {
      label : len(ds) / (len(labels) * counts[i]) for i, label in enumerate(labels) if counts.get(i, 0) > 0
  }

  tot = sum(weights.values())
  norm =  {
      k: round(v / tot, 4) for k,v in weights.items()
  }

  return norm

# Summarization (via Decoder)


In [ ]:
from datasets import ClassLabel

groupings = {
    'DELETE' : set(['ambiguous']),
}

groupings = groupings | {
    label : set([label]) for label in id_to_label.values() if label != 'ambiguous'
}

grouped_ds_train = group_labels(filtered_train_ds, groupings)
grouped_ds_valid = group_labels(filtered_valid_ds, groupings)
grouped_ds_test = group_labels(filtered_test_ds, groupings)


Map:   0%|          | 0/22157 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22157 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/22157 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4083 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4083 [00:00<?, ? examples/s]

Map:   0%|          | 0/9072 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9072 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9072 [00:00<?, ? examples/s]

In [ ]:
id_to_group = { i: group for i, group in enumerate(grouped_ds_train.features['group'].names) }
id_to_group

{0: 'aircraft',
 1: 'airport',
 2: 'airspacestructure',
 3: 'atcequipment/navfacility/buildings',
 4: 'chartorpublication',
 5: 'companypolicy',
 6: 'environment-nonweatherrelated',
 7: 'equipment/tooling',
 8: 'humanfactors',
 9: 'incorrect/notinstalled/unavailablepart',
 10: 'logbookentry',
 11: 'manuals',
 12: 'mel',
 13: 'procedure',
 14: 'softwareandautomation',
 15: 'staffing',
 16: 'weather'}

In [ ]:
from pprint import pprint
pprint(grouped_ds_valid.to_pandas().set_index('acn').loc[[1962404, 1944918, 1902367, 2046198, 1917476]]['text'].to_list())

["Narrative 1 - 'Ramp Agent could not speak English. Did not understand the "
 'security check process. Second issue; Dry Ice loaded in the lower forward '
 'without notification. Only found out about the dry ice and its requirements '
 'when I read the paperwork. Ramp did not pass the message to dispatch or us. '
 'Thankfully no one was affected by the CO2. The ZZZZ ramp seems to be less '
 'professional than before. Things like this are happening more and more. '
 'Please correct. [This issue was caused by the] lack of adherence to '
 'procedures and not caring because there are no consequences. Suggestions: '
 "Follow the procedures.'",
 "Narrative 1 - 'Departing ZZZ I received notification late in the boarding/ "
 'loading process we were taking HAZMAT. Specifically Nuclear Medicine. I '
 'received the amended Release as I was reviewing an MEL. I then spoke to the '
 'Ramp Agents who loaded the HAZMAT. I never received a NOTAC. I think in my '
 'head after talking to the Ramp Agent

### Make Summary Dataset

In [ ]:
def get_df(ds):
  df = ds.to_pandas().set_index('acn')
  df.drop('label', inplace=True, axis=1)
  df.group = df.group.map(id_to_group)
  return df

train_df = get_df(grouped_ds_train)
valid_df = get_df(grouped_ds_valid)
test_df = get_df(grouped_ds_test)

In [ ]:
sys_prompt = 'You are an expert Aviation Safety Analyst  an expert aviation safety analyst and technical writer.'
_template = """

Task: Read the provided aviation safety report and generate a concise, objective synopsis.

Information Requirements:
Your synopsis should be a concise, chronological summary of the facts and critical chain of events leading to the incident, specifying WHAT happened and WHY.

As you construct your response, here are examples of problems you would include in your synopsis (if mentioned):
  - Environment & Infrastructure: Weather, non-weather hazards (wildlife, drones), airport surface (lighting, markings), or airspace design conflicts.
  - Technical & Systems: Aircraft mechanical failures, software/automation glitches (FMS, EFB), ATC facility equipment, or ground tooling.
  - Human & Execution: Cognitive human factors (fatigue, distraction, miscommunication), procedure/SOP non-compliance, or staffing shortages.
  - Policy & Documentation: Company management pressure, chart/manual errors, checklist omissions, or maintenance compliance (MEL, wrong parts, logbooks).

Be sure to detail WHAT specifically happened.
Do not use broad statements to describe the situation (e.g. do NOT say "an environment or infrastructure error." or "there was an issue with policy documentation.").

Constraints & Tone:
* Maintain a strictly objective and professional tone.
* Do not speculate, hallucinate, or introduce any outside information not explicitly stated in the source text.
* Keep the synopsis brief (2 to 5 sentences) and stick strictly to the facts.
* Use clear, standard aviation terminology. If an acronym is used, always use the entire phrase, never use the acronym.
* ALWAYS output the requested JSON, with the report number (as given to you) and the synopsis (which you generate yourself).
* NEVER include introductory or concluding conversational filler.

Return your answer using the following strictly valid JSON format:
{{
  "report_number": "<the report number, exactly as it appears>",
  "synopsis": "<the synopsis you generate>"
}}

Source Text:
REPORT NUMBER: {acn}
REPORT TEXT: {text}
"""

def prompt_template(acn, text):
  return _template.format(acn=acn, text=text)

def make_inference_message(row):
  prompt = prompt_template(acn=row.name, text=row['text'])

  messages = []
  messages.append({'role' : 'system', 'content' : sys_prompt})
  messages.append({'role' : 'user', 'content' : prompt})

  return messages

pprint(make_inference_message(valid_df.iloc[1]))

[{'content': 'You are an expert Aviation Safety Analyst  an expert aviation '
             'safety analyst and technical writer.',
  'role': 'system'},
 {'content': '\n'
             '\n'
             'Task: Read the provided aviation safety report and generate a '
             'concise, objective synopsis. \n'
             '\n'
             'Information Requirements:\n'
             'Your synopsis should be a concise, chronological summary of the '
             'facts and critical chain of events leading to the incident, '
             'specifying WHAT happened and WHY.\n'
             '\n'
             'As you construct your response, here are examples of problems '
             'you would include in your synopsis (if mentioned):\n'
             '  - Environment & Infrastructure: Weather, non-weather hazards '
             '(wildlife, drones), airport surface (lighting, markings), or '
             'airspace design conflicts.\n'
             '  - Technical & Systems: Aircraft mechani

In [ ]:

from datasets import Dataset, DatasetDict, load_dataset, ClassLabel

def make_dataset(df:pd.DataFrame):
  res = pd.DataFrame(df.apply(lambda r : make_inference_message(r), axis=1), columns=['messages'])
  res['label'] = df.group
  res['text'] = df.text
  ds = Dataset.from_pandas(res)
  groups = sorted(list(df.group.unique()))
  ds = ds.cast_column("label", ClassLabel(names=groups))
  return ds

inference_train_ds = make_dataset(train_df)
inference_valid_ds = make_dataset(valid_df)
inference_test_ds = make_dataset(test_df)
inference_valid_ds

Casting the dataset:   0%|          | 0/22157 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4083 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9072 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'label', 'text', 'acn'],
    num_rows: 4083
})

In [ ]:
ds_dict = DatasetDict({
    'train' : inference_train_ds,
    'validation' : inference_valid_ds,
    'test' : inference_test_ds,
})

In [ ]:
ds_dict.push_to_hub("sookiemonster/asrs-summarization")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/23 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  68%|######7   | 33.5MB / 49.6MB            

...

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  87%|########6 | 16.9MB / 19.6MB            

CommitInfo(commit_url='https://huggingface.co/datasets/sookiemonster/asrs-summarization/commit/dc0bea315b9a32b8cc169ea5d2fbe20459f51fa0', commit_message='Upload dataset', commit_description='', oid='dc0bea315b9a32b8cc169ea5d2fbe20459f51fa0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sookiemonster/asrs-summarization', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sookiemonster/asrs-summarization'), pr_revision=None, pr_num=None)

### Load Dataset

In [ ]:
from datasets import load_dataset
repo_id = "sookiemonster/asrs-summarization"

raw_ds = load_dataset(repo_id)
train_ds = raw_ds['train']
train_df = train_ds.to_pandas().set_index('acn')

valid_ds = raw_ds['validation']
valid_df = valid_ds.to_pandas().set_index('acn')


In [ ]:
valid_ds[0]

{'messages': [{'content': 'You are an expert Aviation Safety Analyst  an expert aviation safety analyst and technical writer.',
   'role': 'system'},
  {'content': '\n\nTask: Read the provided aviation safety report and generate a concise, objective synopsis. \n\nInformation Requirements:\nYour synopsis should be a concise, chronological summary of the facts and critical chain of events leading to the incident, specifying WHAT happened and WHY.\n\nAs you construct your response, here are examples of problems you would include in your synopsis (if mentioned):\n  - Environment & Infrastructure: Weather, non-weather hazards (wildlife, drones), airport surface (lighting, markings), or airspace design conflicts.\n  - Technical & Systems: Aircraft mechanical failures, software/automation glitches (FMS, EFB), ATC facility equipment, or ground tooling.\n  - Human & Execution: Cognitive human factors (fatigue, distraction, miscommunication), procedure/SOP non-compliance, or staffing shortages.\

### Inference

In [ ]:
import asyncio
from google.colab import userdata
from openai import AsyncOpenAI

model = "openai/gpt-oss-120b"
sem = asyncio.Semaphore(190) # DI only allows <200 concurrent per model

client = AsyncOpenAI(
    api_key=userdata.get("DEEPINFRA_TOKEN"),
    base_url="https://api.deepinfra.com/v1/openai",
)

async def ask_model(messages):
  async with sem:
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        prompt_cache_key='incident-report-analysis',
        reasoning_effort='low',
        response_format={"type": "json_object"},
    )
    return response

In [ ]:
small_train = train_ds.shuffle(seed=42).select(range(5000))
small_train.to_pandas()['label'].value_counts()

,count
label,
8,1269
13,1262
0,775
16,387
5,293
6,249
4,166
3,146
2,142


In [ ]:
# batch = list(small_train['messages'])
batch = list(valid_ds['messages'])
batch[0]

[{'content': 'You are an expert Aviation Safety Analyst  an expert aviation safety analyst and technical writer.',
  'role': 'system'},
 {'content': '\n\nTask: Read the provided aviation safety report and generate a concise, objective synopsis. \n\nInformation Requirements:\nYour synopsis should be a concise, chronological summary of the facts and critical chain of events leading to the incident, specifying WHAT happened and WHY.\n\nAs you construct your response, here are examples of problems you would include in your synopsis (if mentioned):\n  - Environment & Infrastructure: Weather, non-weather hazards (wildlife, drones), airport surface (lighting, markings), or airspace design conflicts.\n  - Technical & Systems: Aircraft mechanical failures, software/automation glitches (FMS, EFB), ATC facility equipment, or ground tooling.\n  - Human & Execution: Cognitive human factors (fatigue, distraction, miscommunication), procedure/SOP non-compliance, or staffing shortages.\n  - Policy & D

In [ ]:
from tqdm.asyncio import tqdm_asyncio

tasks = [ask_model(p) for p in batch]
# results = await asyncio.gather(*tasks, return_exceptions=True)
results = await tqdm_asyncio.gather(*tasks)

100%|██████████| 4083/4083 [02:32<00:00, 26.69it/s]


In [ ]:
for entry in results:
  print(json.dumps(entry.to_dict(), indent=2))

{
  "id": "chatcmpl-R0WiOfeaitv45Kvr2PpbkNm7",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\n  \"report_number\": \"2018769\",\n  \"synopsis\": \"During a student pilot landing, light variable winds with occasional updrafts and downdrafts over the runway caused the aircraft to encounter a downdraft, resulting in a rapid vertical descent. The student flared too early, and the flight instructor pushed the nose forward to maintain approach speed, but the downdraft increased the descent rate. A go\u2011around was commanded, yet the aircraft contacted the runway hard and the tail impacted the ground during the aborted landing.\"\n}",
        "role": "assistant",
        "tool_calls": [],
        "reasoning_content": "We need JSON. Report number 2018769. Synopsis 2-5 sentences. Include what happened and why: wind variable, dowdraft, early flare, instructor pitch, go-around, tail strike.",
        "name"

In [ ]:
# save[4]

{'id': 'chatcmpl-RfU59lgEnDUtKGGWVHnItQXo',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': '{\n  "report_number": "1507597",\n  "synopsis": "The Jeppesen Professional navigation database flags a radiation warning area extending from CKH to PHNL, indicating that aircraft operating within this airspace at approximately 2800 feet above antenna systems could be exposed to direct radiation potentially causing harmful effects; the report requests clarification of the warning’s meaning and medical assessment of exposure levels and mitigation for flights inbound to Honolulu."\n}',
    'role': 'assistant',
    'tool_calls': [],
    'reasoning_content': "We need JSON. Report number 1507597. Synopsis: Jeppesen Professional shows a radiation warning area from CKH to PHNL at 2800 ft above antenna systems; flights in that corridor may be exposed to direct radiation. Operator requests interpretation and medical exposure info. That's it.",
    'nam

In [ ]:
save = [res.to_dict() for res in results]

output_filename = "oss_120b_validation.json"
with open(output_filename, "w") as f:
  json.dump(save, f, indent=2)

### Load Inference Data

In [ ]:
output_filename = "oss_120b_validation.json"

raw_data = None
with open(output_filename, "r") as f:
  raw_data = json.load(f)

raw_data[0]

{'id': 'chatcmpl-RS1Bp1Ae8O42PDnMLIyGd8Jc',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': '{\n  "report_number": "2018769",\n  "synopsis": "During a training landing, light variable wind with intermittent updrafts and downdrafts caused a sudden vertical descent when the student pilot flared prematurely. The flight instructor pushed the nose forward to maintain approach speed, but a downdraft increased the descent rate, leading the instructor to initiate a go‑around. The aircraft contacted the runway hard, and the tail impacted the ground during the attempted go‑around."\n}',
    'role': 'assistant',
    'tool_calls': [],
    'reasoning_content': 'We need JSON. Provide report number 2018769. Synopsis 2-5 sentences. Include what happened and why: variable wind, updraft/downdraft, student flared early, instructor intervened, aircraft encountered downdraft, large descent, go-around attempted but tail struck during hard touchdown. Provi

In [ ]:
failed = []
def get_generation(result:dict) -> dict:
  try:
    text = result['choices'][0]['message']['content']
    text = json.loads(text)
    return text
  except json.JSONDecodeError:
    print("Failed to decode: ", result)
    failed.append(result)
    return {'report_number' : None, 'synopsis' : None}

def get_usage(result:dict) -> dict:
  return result['usage']

generations = [get_generation(pred) for pred in raw_data]
usages = [get_usage(pred) for pred in raw_data]

In [ ]:
print(len(failed))
# failed[5]['choices'][0]['message']

0


In [ ]:
synopses = pd.DataFrame(generations).rename({'report_number' : 'acn'}, axis=1).set_index('acn')
synopses = synopses.dropna(axis=0)
synopses.index = synopses.index.astype(int)
synopses

,synopsis
acn,
2018769,"During a training landing, light variable wind..."
2185235,A newly‑certified unmanned aircraft operator u...
2156158,"During descent, the aircraft was using only th..."
2139473,During climb out from ZZZ on a revenue flight ...
1813457,"During boarding of flight ZZZ-ZZZ1, a fume eve..."
...,...
2153249,"Approximately 15 minutes after departure, an e..."
1895387,Aircraft X arrived at ZZZ for a scheduled righ...
1939197,A drone was operated within 100 yards of the h...


In [ ]:
from datasets import Dataset, DatasetDict

def make_ds(split:str):
  df = None
  if split == 'train': df = train_df
  elif split == 'validation': df = valid_df
  else: raise "Not an allowed split!"

  joint = synopses.join(df, how='inner').drop(['text', 'messages'], axis=1)
  summary_ds = Dataset.from_pandas(joint)
  summary_ds = summary_ds.cast_column('label', train_ds.features['label'])
  return summary_ds

# summary_ds_valid = make_ds('train')
summary_ds_valid = make_ds('validation')

Casting the dataset:   0%|          | 0/4083 [00:00<?, ? examples/s]

In [ ]:
old = datasets.load_dataset('sookiemonster/asrs-summarization-classification')

README.md:   0%|          | 0.00/909 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4957 [00:00<?, ? examples/s]

In [ ]:
old['validation'] = summary_ds_valid

old

DatasetDict({
    train: Dataset({
        features: ['synopsis', 'label', 'acn'],
        num_rows: 4957
    })
    validation: Dataset({
        features: ['synopsis', 'label', 'acn'],
        num_rows: 4083
    })
})

In [ ]:
old.push_to_hub('sookiemonster/asrs-summarization-classification')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.70MB / 1.70MB            

...

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  39%|###8      |  526kB / 1.36MB            

CommitInfo(commit_url='https://huggingface.co/datasets/sookiemonster/asrs-summarization-classification/commit/3de389fae0900647534c96ef4160790312803dac', commit_message='Upload dataset', commit_description='', oid='3de389fae0900647534c96ef4160790312803dac', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sookiemonster/asrs-summarization-classification', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sookiemonster/asrs-summarization-classification'), pr_revision=None, pr_num=None)

# ModernBERT

In [ ]:
import datasets

raw_ds = datasets.load_dataset('sookiemonster/asrs-summarization-classification')
labels = raw_ds['train'].features['label'].names
id2label = { idx : label for idx, label in enumerate(labels)}
label2id = { label:idx for idx, label in enumerate(labels)}
raw_ds

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4957 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4083 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['synopsis', 'label', 'acn'],
        num_rows: 4957
    })
    validation: Dataset({
        features: ['synopsis', 'label', 'acn'],
        num_rows: 4083
    })
})

In [ ]:
# small_ds = raw_ds['train'].train_test_split(test_size=0.4, seed=42)
train_ds = raw_ds['train']
valid_ds = raw_ds['validation']

### Find Balanced Class Weights

In [ ]:
def get_inv_class_weights(labels, temp=1.0):
  unique, counts = np.unique(labels, return_counts=True)
  weights = (1 / counts) ** (1/temp)
  return np.round(weights / sum(weights), 3)

In [ ]:
weights = get_inv_class_weights(train_ds['label'], temp=4.0)

li = []
for idx, label in enumerate(labels):
  li.append([label, weights[idx]])

display(pd.DataFrame(li))
print(weights)

,0,1
0,aircraft,0.033
1,airport,0.057
2,airspacestructure,0.050
3,atcequipment/navfacility/buildings,0.049
4,chartorpublication,0.048
5,companypolicy,0.042
6,environment-nonweatherrelated,0.043
7,equipment/tooling,0.057
8,humanfactors,0.029
9,incorrect/notinstalled/unavailablepart,0.080


[0.033 0.057 0.05  0.049 0.048 0.042 0.043 0.057 0.029 0.08  0.144 0.071
 0.072 0.029 0.094 0.065 0.039]


In [ ]:
sum(list([float(w) for w in weights]))

1.002

In [ ]:
list([float(w) for w in weights])

[0.033,
 0.057,
 0.05,
 0.049,
 0.048,
 0.042,
 0.043,
 0.057,
 0.029,
 0.08,
 0.144,
 0.071,
 0.072,
 0.029,
 0.094,
 0.065,
 0.039]

In [ ]:
weights = get_inv_class_weights(train_ds['label'], temp=2.0)

li = []
for idx, label in enumerate(labels):
  li.append([label, weights[idx]])

display(pd.DataFrame(li))
# print(weights)
print(sum(list([float(w) for w in weights])))
print(list([float(w) for w in weights]))

,0,1
0,aircraft,0.015
1,airport,0.046
2,airspacestructure,0.034
3,atcequipment/navfacility/buildings,0.034
4,chartorpublication,0.032
5,companypolicy,0.024
6,environment-nonweatherrelated,0.026
7,equipment/tooling,0.045
8,humanfactors,0.011
9,incorrect/notinstalled/unavailablepart,0.089


1.001
[0.015, 0.046, 0.034, 0.034, 0.032, 0.024, 0.026, 0.045, 0.011, 0.089, 0.288, 0.07, 0.072, 0.012, 0.123, 0.059, 0.021]


### Tuning

In [ ]:
import torch
import sys

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}")
print(f"PyTorch Version: {torch.__version__.split('+')[0]}")
print(f"CUDA Version (Torch): {torch.version.cuda}")
print(f"CXX11 ABI: {torch._C._GLIBCXX_USE_CXX11_ABI}")

Python Version: 3.12
PyTorch Version: 2.10.0
CUDA Version (Torch): 12.8
CXX11 ABI: True


In [ ]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 4.9 MB/s eta 0:00:00


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00


In [ ]:
import evaluate
from sklearn.metrics import classification_report, confusion_matrix


accuracy = evaluate.load("accuracy")
precison = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    print(classification_report(labels, predictions))

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision_macro": precison.compute(predictions=predictions, references=labels, zero_division=np.nan, average="macro")["precision"],
        "recall_macro": recall.compute(predictions=predictions, references=labels, average="macro")["recall"]
    }

In [ ]:
from transformers import Trainer
import torch.nn as nn

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32, device="cuda")
        self.loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(torch.device("cuda")))

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss = self.loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "nomic-ai/modernbert-embed-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def preprocess(ds:datasets):
    PREFIX = "classification: "
    res = ds.remove_columns(["acn"]).rename_column('synopsis', 'text')
    res = res.map(lambda examples : {"text" : PREFIX + examples["text"]})
    res = res.map(lambda examples : tokenizer(
        examples["text"],
        padding=True,
        return_tensors="pt"
    ), batched=True)
    res = res.remove_columns(['text'])
    return res

tokenized_train = preprocess(train_ds)
tokenized_eval = preprocess(valid_ds)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/4957 [00:00<?, ? examples/s]

Map:   0%|          | 0/4957 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.bfloat16,
    num_labels=len(id2label),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device=device)

model.safetensors:   0%|          | 0.00/596M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: nomic-ai/modernbert-embed-base
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 
head.norm.weight  | MISSING | 
head.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer

# Adjusted to sum to 1
# TEMP: 4.0
# class_weights = [0.034,
#  0.058,
#  0.052,
#  0.049,
#  0.047,
#  0.041,
#  0.045,
#  0.061,
#  0.029,
#  0.077,
#  0.128,
#  0.075,
#  0.076,
#  0.029,
#  0.094,
#  0.066,
#  0.039]

# TEMP 2.0
class_weights = [0.015, 0.046, 0.034, 0.034, 0.032, 0.024, 0.026, 0.045, 0.011, 0.089, 0.288, 0.07, 0.072, 0.012, 0.123, 0.059, 0.021]

args = TrainingArguments(
    output_dir="./modernbert-finetuned",
    bf16=True,

    learning_rate=3e-5,
    lr_scheduler_type="cosine_with_restarts",
    lr_scheduler_kwargs={"num_cycles":4},
    num_train_epochs=15,
    weight_decay=0.01,

    per_device_train_batch_size=512,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=True,
    hub_model_id="sookiemonster/asrs-modernbert-summary-classifier-cosine-restarts",

    report_to="none",
    logging_steps=50,
)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [ ]:
# Cosine w/ 4-cycles -- num epochs 15 -- LR 3e-5
# Train on 5_000 train & eval on full validation
# Temp 2.0
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro
1,No log,1.433255,0.566985,0.263535,0.451312
2,No log,1.309604,0.624051,0.319764,0.411526
3,No log,1.310157,0.620622,0.322359,0.427148
4,No log,1.380689,0.617683,0.297481,0.443373
5,1.290355,1.335602,0.605192,0.294149,0.419328
6,1.290355,1.318889,0.619642,0.315413,0.459975
7,1.290355,1.328175,0.618173,0.312141,0.479415
8,1.290355,1.376359,0.603233,0.335347,0.410718
9,1.290355,1.339398,0.627970,0.321070,0.468100
10,1.074605,1.347061,0.618173,0.309807,0.482169


              precision    recall  f1-score   support

           0       0.88      0.78      0.83      1489
           1       0.56      0.45      0.50       110
           2       0.10      0.59      0.17        58
           3       0.26      0.65      0.37        57
           4       0.37      0.66      0.47        61
           5       0.05      0.17      0.08        24
           6       0.19      0.36      0.25       107
           7       0.23      0.53      0.32        17
           8       0.76      0.45      0.56      1517
           9       0.00      0.00      0.00         9
          10       0.00      0.00      0.00         1
          11       0.06      1.00      0.11         4
          12       0.11      0.09      0.10        11
          13       0.26      0.28      0.27       435
          14       0.07      0.03      0.04        30
          15       0.11      0.80      0.19         5
          16       0.48      0.83      0.61       148

    accuracy              

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.85      0.84      0.84      1489
           1       0.49      0.49      0.49       110
           2       0.20      0.33      0.25        58
           3       0.30      0.67      0.42        57
           4       0.40      0.66      0.50        61
           5       0.04      0.21      0.07        24
           6       0.20      0.29      0.24       107
           7       0.19      0.65      0.30        17
           8       0.75      0.58      0.65      1517
           9       0.22      0.67      0.33         9
          10       0.00      0.00      0.00         1
          11       0.20      0.25      0.22         4
          12       0.06      0.18      0.10        11
          13       0.27      0.24      0.25       435
          14       0.00      0.00      0.00        30
          15       0.06      0.20      0.10         5
          16       0.56      0.76      0.64       148

    accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.88      0.81      0.84      1489
           1       0.48      0.50      0.49       110
           2       0.15      0.33      0.21        58
           3       0.31      0.67      0.42        57
           4       0.49      0.59      0.54        61
           5       0.01      0.04      0.02        24
           6       0.21      0.35      0.26       107
           7       0.18      0.59      0.27        17
           8       0.74      0.61      0.67      1517
           9       0.29      0.44      0.35         9
          10       0.00      0.00      0.00         1
          11       0.14      0.75      0.24         4
          12       0.10      0.27      0.15        11
          13       0.27      0.25      0.26       435
          14       0.12      0.03      0.05        30
          15       0.10      0.60      0.18         5
          16       0.53      0.80      0.64       148

    accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=1.109230982462565, metrics={'train_runtime': 715.7138, 'train_samples_per_second': 103.889, 'train_steps_per_second': 0.21, 'total_flos': 1.207594337004036e+16, 'train_loss': 1.109230982462565, 'epoch': 15.0})

### Temp 4.0 & Batch (256)

In [ ]:
# Cosine w/ 2-cycles -- num epochs 5 -- LR 3e-5
# Train on 5_000 train & eval on full validation
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro
1,No log,1.467750,0.595151,0.449551,0.249838
2,No log,1.342046,0.613030,0.369260,0.253369
3,1.763269,1.245257,0.651482,0.333266,0.357909
4,1.763269,1.213783,0.641930,0.329526,0.368949
5,1.343846,1.208854,0.641685,0.357360,0.372750


              precision    recall  f1-score   support

           0       0.85      0.73      0.78      1489
           1       1.00      0.05      0.10       110
           2       0.00      0.00      0.00        58
           3       0.33      0.39      0.36        57
           4       0.42      0.36      0.39        61
           5       0.08      0.25      0.12        24
           6       0.31      0.07      0.12       107
           7       0.25      0.12      0.16        17
           8       0.65      0.66      0.65      1517
           9       0.00      0.00      0.00         9
          10       0.00      0.00      0.00         1
          11       1.00      0.25      0.40         4
          12       0.00      0.00      0.00        11
          13       0.21      0.40      0.28       435
          14       0.00      0.00      0.00        30
          15       0.25      0.20      0.22         5
          16       0.50      0.78      0.61       148

    accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.85      0.80      0.82      1489
           1       0.81      0.27      0.41       110
           2       0.11      0.07      0.09        58
           3       0.28      0.44      0.34        57
           4       0.44      0.44      0.44        61
           5       0.07      0.29      0.12        24
           6       0.27      0.12      0.17       107
           7       0.20      0.06      0.09        17
           8       0.70      0.60      0.65      1517
           9       0.00      0.00      0.00         9
          10       0.00      0.00      0.00         1
          11       0.00      0.00      0.00         4
          12       0.00      0.00      0.00        11
          13       0.23      0.39      0.29       435
          14       0.00      0.00      0.00        30
          15       0.00      0.00      0.00         5
          16       0.47      0.82      0.60       148

    accuracy              

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.87      0.80      0.83      1489
           1       0.59      0.45      0.51       110
           2       0.20      0.29      0.24        58
           3       0.29      0.65      0.40        57
           4       0.52      0.48      0.50        61
           5       0.05      0.17      0.08        24
           6       0.24      0.22      0.23       107
           7       0.26      0.41      0.32        17
           8       0.71      0.66      0.69      1517
           9       0.33      0.11      0.17         9
          10       0.00      0.00      0.00         1
          11       0.25      0.50      0.33         4
          12       0.12      0.09      0.11        11
          13       0.27      0.30      0.28       435
          14       0.00      0.00      0.00        30
          15       0.12      0.40      0.18         5
          16       0.53      0.80      0.64       148

    accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=1.5535578155517578, metrics={'train_runtime': 233.2294, 'train_samples_per_second': 106.269, 'train_steps_per_second': 0.429, 'total_flos': 4025314456680120.0, 'train_loss': 1.5535578155517578, 'epoch': 5.0})

In [ ]:
# Cosine w/ 3-cycles -- num epochs 15 -- LR 3e-5
# Train on subset of train (60/40 train-validation split)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro
1,No log,1.525070,0.544125,0.455416,0.368251
2,No log,1.466919,0.555219,0.438850,0.392439
3,No log,1.465346,0.551689,0.437617,0.387127
4,No log,1.473207,0.540595,0.409565,0.368761
5,1.172116,1.471373,0.543621,0.414315,0.369824
6,1.172116,1.557569,0.523954,0.379585,0.395905
7,1.172116,1.520105,0.535048,0.383826,0.369836
8,1.172116,1.494668,0.534544,0.387368,0.382155
9,1.016864,1.494248,0.546142,0.410405,0.372356
10,1.016864,1.494975,0.545134,0.410048,0.372529


              precision    recall  f1-score   support

           0       0.63      0.85      0.72       292
           1       0.48      0.30      0.37        33
           2       0.43      0.19      0.27        68
           3       0.45      0.70      0.55        53
           4       0.33      0.47      0.39        57
           5       0.42      0.45      0.43       106
           6       0.58      0.16      0.25       111
           7       0.33      0.02      0.04        42
           8       0.58      0.65      0.62       512
           9       0.40      0.33      0.36         6
          11       0.00      0.00      0.00        17
          12       0.50      0.06      0.11        16
          13       0.48      0.43      0.45       488
          14       0.00      0.00      0.00         4
          15       0.59      0.50      0.54        20
          16       0.62      0.76      0.69       158

    accuracy                           0.54      1983
   macro avg       0.43   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.67      0.82      0.74       292
           1       0.38      0.27      0.32        33
           2       0.36      0.24      0.29        68
           3       0.54      0.62      0.58        53
           4       0.39      0.49      0.44        57
           5       0.43      0.47      0.45       106
           6       0.53      0.28      0.37       111
           7       0.57      0.10      0.16        42
           8       0.57      0.69      0.63       512
           9       0.08      0.17      0.11         6
          11       0.20      0.12      0.15        17
          12       0.21      0.25      0.23        16
          13       0.53      0.41      0.46       488
          14       0.00      0.00      0.00         4
          15       0.46      0.60      0.52        20
          16       0.65      0.76      0.70       158

    accuracy                           0.56      1983
   macro avg       0.41   

...

              precision    recall  f1-score   support

           0       0.69      0.82      0.75       292
           1       0.29      0.30      0.30        33
           2       0.34      0.22      0.27        68
           3       0.52      0.64      0.58        53
           4       0.39      0.49      0.44        57
           5       0.38      0.48      0.43       106
           6       0.43      0.33      0.38       111
           7       0.29      0.10      0.14        42
           8       0.59      0.59      0.59       512
           9       0.00      0.00      0.00         6
          11       0.20      0.06      0.09        17
          12       0.17      0.06      0.09        16
          13       0.48      0.48      0.48       488
          14       0.00      0.00      0.00         4
          15       0.50      0.35      0.41        20
          16       0.72      0.68      0.70       158

    accuracy                           0.54      1983
   macro avg       0.37   

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.69      0.82      0.75       292
           1       0.30      0.30      0.30        33
           2       0.33      0.22      0.26        68
           3       0.52      0.64      0.57        53
           4       0.39      0.49      0.44        57
           5       0.39      0.48      0.43       106
           6       0.42      0.32      0.36       111
           7       0.29      0.10      0.14        42
           8       0.59      0.59      0.59       512
           9       0.00      0.00      0.00         6
          11       0.33      0.12      0.17        17
          12       0.17      0.06      0.09        16
          13       0.48      0.48      0.48       488
          14       0.00      0.00      0.00         4
          15       0.50      0.35      0.41        20
          16       0.71      0.68      0.70       158

    accuracy                           0.54      1983
   macro avg       0.38   

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=180, training_loss=0.9895760748121474, metrics={'train_runtime': 384.1798, 'train_samples_per_second': 116.118, 'train_steps_per_second': 0.469, 'total_flos': 6473062201312440.0, 'train_loss': 0.9895760748121474, 'epoch': 15.0})

In [ ]:
# Cosine -- num epochs 6 -- LR 3e-5
# Train on subset of train (60/40 train-validation split)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
W0506 13:07:11.561000 5224 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro
1,No log,1.837510,0.452345,0.408950,0.225748
2,No log,1.583631,0.515381,0.392976,0.308960
3,No log,1.526907,0.532526,0.456302,0.324739
4,No log,1.499674,0.546646,0.491354,0.343808
5,1.600245,1.491047,0.547151,0.438683,0.347878
6,1.600245,1.491082,0.547151,0.449119,0.348340


              precision    recall  f1-score   support

           0       0.47      0.88      0.62       292
           1       1.00      0.09      0.17        33
           2       0.22      0.03      0.05        68
           3       0.50      0.09      0.16        53
           4       0.30      0.37      0.33        57
           5       0.31      0.42      0.36       106
           6       0.26      0.06      0.10       111
           7       0.00      0.00      0.00        42
           8       0.56      0.40      0.46       512
           9       0.00      0.00      0.00         6
          11       0.00      0.00      0.00        17
          12       0.00      0.00      0.00        16
          13       0.43      0.47      0.45       488
          14       0.00      0.00      0.00         4
          15       0.00      0.00      0.00        20
          16       0.44      0.80      0.57       158

    accuracy                           0.45      1983
   macro avg       0.28   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.67      0.81      0.74       292
           1       0.43      0.27      0.33        33
           2       0.36      0.22      0.27        68
           3       0.47      0.62      0.54        53
           4       0.37      0.46      0.41        57
           5       0.44      0.43      0.44       106
           6       0.47      0.23      0.30       111
           7       0.75      0.07      0.13        42
           8       0.58      0.63      0.61       512
           9       0.14      0.17      0.15         6
          11       0.00      0.00      0.00        17
          12       0.50      0.06      0.11        16
          13       0.48      0.49      0.48       488
          14       0.00      0.00      0.00         4
          15       0.44      0.35      0.39        20
          16       0.63      0.76      0.69       158

    accuracy                           0.55      1983
   macro avg       0.42   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=72, training_loss=1.5004279878404405, metrics={'train_runtime': 164.9784, 'train_samples_per_second': 108.16, 'train_steps_per_second': 0.436, 'total_flos': 2589224880524976.0, 'train_loss': 1.5004279878404405, 'epoch': 6.0})